# 📊 구독 해지 예측 모델 (Subscription Churn Prediction)
본 노트북은 `mock_data.csv` 데이터를 기반으로 다음 달 구독을 유지할지(1) 해지할지(0)를 예측하는 초기 모델(Logistic Regression)을 학습하는 코드입니다.

- 0: 해지 (Churn)
- 1: 유지 (Retain)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
plt.rc('font', family='Malgun Gothic') # 윈도우 한글 깨짐 방지
plt.rc('axes', unicode_minus=False)

## 1. 데이터 로드 (Load Data)
사전에 생성된 `mock_data.csv` 파일(5,000건)을 불러옵니다.

In [ ]:
df = pd.read_csv('mock_data.csv')

display(df.head())
print("\n[타겟 분포 (유지 1, 해지 0)]")
print(df['Target'].value_counts())

## 2. 파생 변수 생성
- **Inquiry_Rate (문의율)** = Customer_Inquiry / (Access_Days + 1)
- **Value_Score (가성비 점수)** = Usage_Time / Monthly_Fee

In [ ]:
df['Inquiry_Rate'] = df['Customer_Inquiry'] / (df['Access_Days'] + 1)
df['Value_Score'] = df['Usage_Time'] / df['Monthly_Fee']

display(df.head())
print("데이터 크기:", df.shape)

## 3. 전처리 및 데이터 분할
기본 로지스틱 회귀 모델 학습을 위해 특성들의 스케일을 맞춰줍니다 (표준화).

In [ ]:
X = df.drop('Target', axis=1)
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. 로지스틱 회귀 모델 학습

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1] # 클래스 1(유지)에 대한 확률

## 5. 상세 모델 성능 평가
단순 Accuracy(정확도)가 아닌, 클래스 0(해지) 포착 관점에서의 성능 평가를 수행합니다.

In [ ]:
# pos_label=0 (해지)를 양성 클래스로 지정하여 정밀도, 재현율, F1을 계산합니다.
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=0)
recall = recall_score(y_test, y_pred, pos_label=0)
f1 = f1_score(y_test, y_pred, pos_label=0)
roc_auc = roc_auc_score(y_test, y_proba)

print("=== 로지스틱 회귀 초기 모델 평가 결과 ===")
print(f"정확도 (Accuracy)  : {accuracy:.4f}")
print(f"정밀도 (Precision) (클래스 0 기준) : {precision:.4f}")
print(f"재현율 (Recall)    (클래스 0 기준) : {recall:.4f}")
print(f"F1-점수 (F1-Score) (클래스 0 기준) : {f1:.4f}")
print(f"ROC-AUC 점수      : {roc_auc:.4f}")
print("===========================================")

print("\n[분류 보고서 (Classification Report)]")
print(classification_report(y_test, y_pred))

## 6. 혼동 행렬 (Confusion Matrix)
예측 결과를 눈으로 쉽게 확인하기 위한 시각화

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['해지예측(0)', '유지예측(1)'], 
            yticklabels=['실제해지(0)', '실제유지(1)'])
plt.title('혼동 행렬 (Confusion Matrix)')
plt.ylabel('실제 값 (True Label)')
plt.xlabel('예측 값 (Predicted Label)')
plt.show()